In [23]:
import pandas as pd

df = pd.read_csv("../data/raw/train.csv")
df.head()
original_shape = df.shape
print("Original shape:", original_shape)

Original shape: (17307, 3)


In [3]:
print(df.shape)
print(df.columns)

(17307, 3)
Index(['essay_id', 'full_text', 'score'], dtype='str')


## Data Ingestion 
The dataset was loaded from the Kaggle Learning Agency Lab Automated Essay Scoring 2.0 competition. The training dataset (train.csv) contains essay text and corresponding human-assigned scores, which will be used for analysis.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17307 entries, 0 to 17306
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   essay_id   17307 non-null  str  
 1   full_text  17307 non-null  str  
 2   score      17307 non-null  int64
dtypes: int64(1), str(2)
memory usage: 405.8 KB


In [5]:
df.isnull().sum()

essay_id     0
full_text    0
score        0
dtype: int64

In [6]:
df.describe()

,score
count,17307.000000
mean,2.948402
std,1.044899
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,6.000000


## Data Cleaning


In [19]:
# Remove duplicates
df = df.drop_duplicates(subset=['full_text'])

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)                 # remove line breaks
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)      # remove special characters
    text = re.sub(r'\s+', ' ', text)                # remove extra spaces
    text = text.strip()
    return text

df['clean_text'] = df['full_text'].apply(clean_text)
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

# Remove empty cleaned text if any
df = df[df['clean_text'] != '']

# Remove very short essays
df = df[df['word_count'] > 20]

print("Shape after cleaning:", df.shape)
df[['essay_id', 'full_text', 'clean_text', 'word_count']].head()

Shape after cleaning: (17307, 5)


,essay_id,full_text,clean_text,word_count
0,000d118,Many people have car where they live. The thin...,many people have car where they live the thing...,498
1,000fe60,I am a scientist at NASA that is discussing th...,i am a scientist at nasa that is discussing th...,332
2,001ab80,People always wish they had the same technolog...,people always wish they had the same technolog...,550
3,001bdc0,"We all heard about Venus, the planet without a...",we all heard about venus the planet without al...,448
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",dear state senator this is a letter to argue i...,373


In [22]:
print("Shape after cleaning:", df.shape)

Shape after cleaning: (17307, 5)


The dataset did not contain missing values, duplicate essay texts, or essays below the chosen minimum word threshold. However, data cleaning was still performed to standardize the text by normalizing case, removing line breaks, removing special characters, and trimming whitespace. This prepared the data for feature engineering and downstream analysis.

In [20]:
df.to_csv("../data/processed/essays_cleaned.csv", index=False)